# Participant-Independent WAV Feature Dataset Builder

This notebook builds the next-stage machine-learning dataset directly from the raw WAV files in the supplied Google Drive folder.

### Frozen protocol
- 401 WAV recordings
- 59 participants
- Three source class codes: `H`, `L`, `P`
- Source label mapping is verified from `file_labels.csv`: `H → 0`, `L → 1`, `P → 2`
- Frozen participant-independent split:
  - Train: 41 participants, 281 recordings
  - Validation: 9 participants, 60 recordings
  - Test: 9 participants, 60 recordings
- No participant appears in more than one split.

### Features generated
1. Audio quality-control metadata
2. Fixed-length 128-frame temporal acoustic representation
3. Log-Mel, MFCC, delta-MFCC, energy, spectral and pitch/voicing features
4. Handcrafted global acoustic descriptors
5. HuBERT pretrained speech embeddings
6. Train-only PCA reduction of HuBERT embeddings
7. Train-only feature standardization
8. Participant-grouped 5-fold cross-validation manifest
9. Reproducibility hashes, integrity audit and plots

The notebook does **not** infer semantic meanings for H, L or P. It preserves the source coding exactly.

In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
import sys, subprocess
packages = [
    "librosa", "soundfile", "transformers", "accelerate",
    "scikit-learn", "joblib", "tqdm"
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", *packages]
)
print("Dependencies installed.")


In [ ]:
# ============================================================
# 2. Mount Google Drive and imports
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, math, hashlib, warnings, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.stats import iqr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedGroupKFold
import joblib

import torch
from transformers import Wav2Vec2FeatureExtractor, HubertModel

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 3. Configuration

`SOURCE_ROOT_OVERRIDE` can be set manually if automatic discovery finds more than one matching folder. Otherwise leave it as `None`.

In [ ]:
# ============================================================
# 3. Configuration
# ============================================================
MYDRIVE = Path("/content/drive/MyDrive")

# Optional manual override, for example:
# SOURCE_ROOT_OVERRIDE = Path("/content/drive/MyDrive/MyAudioDataset")
SOURCE_ROOT_OVERRIDE = None

OUTPUT_ROOT = MYDRIVE / "Audio_Engagement_Feature_Dataset_PI"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SR = 16000
N_FFT = 400
WIN_LENGTH = 400
HOP_LENGTH = 160
N_MELS = 64
N_MFCC = 20
TEMPORAL_FRAMES = 128

FMIN = 50
FMAX = 7600
PYIN_FMIN = 65
PYIN_FMAX = 500

EXTRACT_HUBERT = True
HUBERT_MODEL_NAME = "facebook/hubert-base-ls960"
HUBERT_CHUNK_SECONDS = 8.0
HUBERT_PCA_DIM = 256

RANDOM_STATE = 42
N_CV_FOLDS = 5

EXPECTED_SAMPLES = 401
EXPECTED_PARTICIPANTS = 59
EXPECTED_CLASS_MAPPING = {"H": 0, "L": 1, "P": 2}

TRAIN_PARTICIPANTS = [3,4,5,6,7,8,9,10,11,13,15,16,17,18,19,23,24,26,27,28,29,30,31,32,33,35,36,37,38,39,42,43,44,45,49,50,52,54,55,57,59]
VAL_PARTICIPANTS   = [1,12,20,25,34,47,48,56,58]
TEST_PARTICIPANTS  = [2,14,21,22,40,41,46,51,53]

EXPECTED_SPLIT_SHA256 = "df860201d0c2ea8a4985fe219043b9c50c5e69723b0012d6caa90800cfa2d8c7"

print("Output:", OUTPUT_ROOT)

In [ ]:
# ============================================================
# 4. Locate source folder automatically
# ============================================================
def find_source_roots(base: Path):
    roots = []
    for label_path in base.rglob("file_labels.csv"):
        parent = label_path.parent
        wav_candidates = [
            parent / "Wav_Files",
            parent / "Wav_Files" / "Wav_Files",
        ]
        if any(p.exists() for p in wav_candidates):
            roots.append(parent)
    return sorted(set(roots))

if SOURCE_ROOT_OVERRIDE is not None:
    SOURCE_ROOT = Path(SOURCE_ROOT_OVERRIDE)
else:
    candidates = find_source_roots(MYDRIVE)
    print("Candidate source roots:")
    for c in candidates:
        print(" -", c)
    if len(candidates) == 0:
        raise FileNotFoundError(
            "Could not locate a folder containing file_labels.csv and Wav_Files. "
            "Set SOURCE_ROOT_OVERRIDE manually."
        )
    if len(candidates) > 1:
        raise RuntimeError(
            "Multiple candidate source folders found. Set SOURCE_ROOT_OVERRIDE "
            "to the intended folder."
        )
    SOURCE_ROOT = candidates[0]

LABEL_CSV = SOURCE_ROOT / "file_labels.csv"

# Locate deepest Wav_Files directory containing WAV files.
wav_dirs = [p for p in SOURCE_ROOT.rglob("Wav_Files") if p.is_dir()]
wav_dirs = [p for p in wav_dirs if any(p.glob("*.wav"))]
if not wav_dirs:
    # recurse one level more generally
    wav_dirs = []
    for p in SOURCE_ROOT.rglob("*"):
        if p.is_dir() and any(p.glob("*.wav")):
            wav_dirs.append(p)

if not wav_dirs:
    raise FileNotFoundError("No WAV directory found under source root.")

# Prefer directory with the most WAV files.
WAV_DIR = max(wav_dirs, key=lambda p: len(list(p.glob("*.wav"))))

print("SOURCE_ROOT:", SOURCE_ROOT)
print("LABEL_CSV:", LABEL_CSV)
print("WAV_DIR:", WAV_DIR)
print("WAV count:", len(list(WAV_DIR.glob("*.wav"))))

In [ ]:
# ============================================================
# 5. Read labels and reconstruct frozen participant split
# ============================================================
labels = pd.read_csv(LABEL_CSV)
labels.columns = [c.strip() for c in labels.columns]

if not {"filename", "label"}.issubset(labels.columns):
    raise ValueError("file_labels.csv must contain columns: filename, label")

def parse_filename(path_string):
    name = os.path.basename(str(path_string)).strip()
    m = re.match(r"(?i)^st(\d+)-(\d+)-([HLP])\.wav$", name)
    if not m:
        raise ValueError(f"Unrecognized WAV filename pattern: {name}")
    participant, trial, code = m.groups()
    return int(participant), int(trial), code.upper(), name

parsed = labels["filename"].apply(parse_filename)
parsed_df = pd.DataFrame(
    parsed.tolist(),
    columns=["participant", "trial", "class_code", "audio_filename"]
)
labels = pd.concat([labels.reset_index(drop=True), parsed_df], axis=1)
labels["label"] = labels["label"].astype(int)

# Verify source label mapping rather than assuming it.
mapping_check = labels.groupby("class_code")["label"].nunique()
if not (mapping_check == 1).all():
    raise ValueError("Class-code to numeric-label mapping is inconsistent in file_labels.csv")

source_mapping = labels.groupby("class_code")["label"].first().to_dict()
print("Source class mapping:", source_mapping)

if source_mapping != EXPECTED_CLASS_MAPPING:
    raise AssertionError(
        f"Unexpected source mapping. Expected {EXPECTED_CLASS_MAPPING}, got {source_mapping}"
    )

split_map = {}
for p in TRAIN_PARTICIPANTS: split_map[p] = "train"
for p in VAL_PARTICIPANTS: split_map[p] = "val"
for p in TEST_PARTICIPANTS: split_map[p] = "test"

labels["split"] = labels["participant"].map(split_map)

if labels["split"].isna().any():
    missing = sorted(labels.loc[labels["split"].isna(), "participant"].unique())
    raise ValueError(f"Participants missing from frozen split: {missing}")

# Map actual files case-insensitively.
wav_paths = list(WAV_DIR.glob("*.wav"))
wav_lookup = defaultdict(list)
for p in wav_paths:
    wav_lookup[p.name.lower()].append(p)

resolved = []
ambiguous = []
missing = []

for name in labels["audio_filename"]:
    matches = wav_lookup.get(name.lower(), [])
    if len(matches) == 1:
        resolved.append(str(matches[0]))
    elif len(matches) == 0:
        resolved.append(None)
        missing.append(name)
    else:
        resolved.append(None)
        ambiguous.append((name, [str(x) for x in matches]))

labels["audio_path"] = resolved

print("Rows:", len(labels))
print("Participants:", labels["participant"].nunique())
print("Missing WAVs:", len(missing))
print("Ambiguous WAVs:", len(ambiguous))

assert len(labels) == EXPECTED_SAMPLES
assert labels["participant"].nunique() == EXPECTED_PARTICIPANTS
assert len(missing) == 0
assert len(ambiguous) == 0

labels["sample_id"] = [
    f"st{p:02d}_trial{t:02d}_{c}"
    for p, t, c in zip(labels["participant"], labels["trial"], labels["class_code"])
]

labels = labels.sort_values(
    ["split", "participant", "trial", "class_code"]
).reset_index(drop=True)

labels.head()

In [ ]:
# ============================================================
# 6. Verify frozen split hash and integrity
# ============================================================
participant_split = pd.DataFrame(
    [(p, split_map[p]) for p in sorted(split_map)],
    columns=["participant", "split"]
)

split_text = participant_split.to_csv(index=False)
split_sha = hashlib.sha256(split_text.encode("utf-8")).hexdigest()

print("Frozen split SHA256:", split_sha)
assert split_sha == EXPECTED_SPLIT_SHA256

train_people = set(TRAIN_PARTICIPANTS)
val_people = set(VAL_PARTICIPANTS)
test_people = set(TEST_PARTICIPANTS)

assert train_people.isdisjoint(val_people)
assert train_people.isdisjoint(test_people)
assert val_people.isdisjoint(test_people)

summary = (
    labels.groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "val", "test"])
)
print(summary)
print("\nParticipants by split:")
print(labels.groupby("split")["participant"].nunique().reindex(["train","val","test"]))

labels.to_csv(OUTPUT_ROOT / "master_manifest.csv", index=False)
participant_split.to_csv(OUTPUT_ROOT / "participant_split.csv", index=False)

for split in ["train", "val", "test"]:
    labels[labels["split"] == split].to_csv(
        OUTPUT_ROOT / f"{split}_manifest.csv", index=False
    )

## 7. Audio loading and quality control

The original waveform amplitude is retained. Audio is converted to mono and resampled to 16 kHz only for feature extraction.

In [ ]:
# ============================================================
# 7. Audio loader and QC
# ============================================================
def load_audio(path, target_sr=SR):
    y, sr = sf.read(path, always_2d=False)
    y = np.asarray(y)

    channels = 1 if y.ndim == 1 else y.shape[1]
    if y.ndim > 1:
        y = np.mean(y, axis=1)

    y = y.astype(np.float32, copy=False)

    # Handle integer-scaled or unusual input robustly.
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    if peak > 1.5:
        y = y / max(peak, 1e-8)

    original_sr = int(sr)
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    return y.astype(np.float32), int(sr), original_sr, channels

def audio_qc(path):
    info = sf.info(path)
    y, sr = sf.read(path, always_2d=False)
    y = np.asarray(y)
    channels = 1 if y.ndim == 1 else y.shape[1]
    if y.ndim > 1:
        mono = np.mean(y, axis=1)
    else:
        mono = y

    mono = mono.astype(np.float32, copy=False)
    peak = float(np.max(np.abs(mono))) if len(mono) else 0.0
    rms = float(np.sqrt(np.mean(np.square(mono))) if len(mono) else 0.0)
    clipping_ratio = float(np.mean(np.abs(mono) >= 0.999)) if len(mono) else 0.0

    # Silence proxy based on relative RMS windows.
    frame_rms = librosa.feature.rms(
        y=mono, frame_length=min(2048, max(256, len(mono))),
        hop_length=512
    ).flatten()
    if len(frame_rms):
        thr = max(np.percentile(frame_rms, 20) * 0.5, 1e-6)
        silence_ratio = float(np.mean(frame_rms <= thr))
    else:
        silence_ratio = 1.0

    return {
        "original_sr": int(info.samplerate),
        "channels": int(channels),
        "frames": int(info.frames),
        "duration_s": float(info.duration),
        "peak_abs": peak,
        "rms_full": rms,
        "clipping_ratio": clipping_ratio,
        "silence_ratio_proxy": silence_ratio,
    }

qc_rows = []
for row in tqdm(labels.itertuples(index=False), total=len(labels), desc="Audio QC"):
    q = audio_qc(row.audio_path)
    q.update({
        "sample_id": row.sample_id,
        "participant": row.participant,
        "trial": row.trial,
        "class_code": row.class_code,
        "label": row.label,
        "split": row.split,
        "audio_filename": row.audio_filename,
    })
    qc_rows.append(q)

qc = pd.DataFrame(qc_rows)
qc.to_csv(OUTPUT_ROOT / "audio_qc.csv", index=False)
qc.describe(include="all")

In [ ]:
# ============================================================
# 8. Dataset overview plots
# ============================================================
fig = plt.figure(figsize=(8,5))
plot_df = labels.groupby(["split","label"]).size().unstack(fill_value=0).reindex(["train","val","test"])
plot_df.plot(kind="bar", ax=plt.gca())
plt.title("Participant-independent class distribution")
plt.xlabel("Split")
plt.ylabel("Number of recordings")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "class_distribution.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
for split in ["train","val","test"]:
    vals = qc.loc[qc["split"] == split, "duration_s"]
    plt.hist(vals, bins=20, alpha=0.45, label=split)
plt.title("Recording duration distribution")
plt.xlabel("Duration (s)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "duration_distribution.png", dpi=300)
plt.show()

## 9. Temporal feature extraction

Each recording is represented as exactly **128 time steps**. The temporal tensor contains:

- 64 log-Mel channels
- 20 MFCC channels
- 20 delta-MFCC channels
- RMS energy
- zero-crossing rate
- spectral centroid
- spectral bandwidth
- spectral rolloff
- spectral flatness
- log-F0
- voiced/unvoiced indicator

Total: **112 temporal channels**.

In [ ]:
# ============================================================
# 9. Temporal feature helpers
# ============================================================
def resize_time(mat, target_frames=TEMPORAL_FRAMES):
    # Input shape: channels x time
    mat = np.asarray(mat, dtype=np.float32)
    if mat.ndim == 1:
        mat = mat[None, :]
    if mat.shape[1] == target_frames:
        return mat
    if mat.shape[1] <= 1:
        return np.repeat(mat, target_frames, axis=1)

    old_x = np.linspace(0.0, 1.0, mat.shape[1])
    new_x = np.linspace(0.0, 1.0, target_frames)
    out = np.vstack([
        np.interp(new_x, old_x, ch) for ch in mat
    ])
    return out.astype(np.float32)

TEMPORAL_FEATURE_NAMES = (
    [f"logmel_{i:02d}" for i in range(N_MELS)] +
    [f"mfcc_{i:02d}" for i in range(N_MFCC)] +
    [f"delta_mfcc_{i:02d}" for i in range(N_MFCC)] +
    [
        "rms", "zcr", "spectral_centroid", "spectral_bandwidth",
        "spectral_rolloff85", "spectral_flatness",
        "log_f0", "voiced"
    ]
)

assert len(TEMPORAL_FEATURE_NAMES) == 112

def safe_pyin(y):
    try:
        f0, voiced_flag, voiced_prob = librosa.pyin(
            y,
            fmin=PYIN_FMIN,
            fmax=PYIN_FMAX,
            sr=SR,
            frame_length=1024,
            hop_length=HOP_LENGTH,
        )
        if f0 is None:
            raise ValueError("pyin returned None")
        f0 = np.asarray(f0, dtype=np.float32)
        voiced = np.asarray(voiced_flag, dtype=np.float32)
        f0 = np.nan_to_num(f0, nan=0.0, posinf=0.0, neginf=0.0)
        voiced = np.nan_to_num(voiced, nan=0.0)
        log_f0 = np.zeros_like(f0, dtype=np.float32)
        mask = f0 > 0
        log_f0[mask] = np.log1p(f0[mask])
        return log_f0, voiced
    except Exception:
        n = max(1, 1 + int(max(0, len(y)-1024) / HOP_LENGTH))
        return np.zeros(n, dtype=np.float32), np.zeros(n, dtype=np.float32)

def extract_temporal(y):
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH, n_mels=N_MELS,
        fmin=FMIN, fmax=FMAX, power=2.0
    )
    logmel = librosa.power_to_db(mel + 1e-10, ref=np.max)

    mfcc = librosa.feature.mfcc(
        S=logmel, n_mfcc=N_MFCC
    )
    delta = librosa.feature.delta(mfcc, width=9, order=1)

    rms = librosa.feature.rms(
        y=y, frame_length=N_FFT, hop_length=HOP_LENGTH
    )
    zcr = librosa.feature.zero_crossing_rate(
        y, frame_length=N_FFT, hop_length=HOP_LENGTH
    )
    centroid = librosa.feature.spectral_centroid(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    bandwidth = librosa.feature.spectral_bandwidth(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    rolloff = librosa.feature.spectral_rolloff(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        roll_percent=0.85
    )
    flatness = librosa.feature.spectral_flatness(
        y=y, n_fft=N_FFT, hop_length=HOP_LENGTH
    )

    log_f0, voiced = safe_pyin(y)

    parts = [
        resize_time(logmel),
        resize_time(mfcc),
        resize_time(delta),
        resize_time(rms),
        resize_time(zcr),
        resize_time(centroid),
        resize_time(bandwidth),
        resize_time(rolloff),
        resize_time(flatness),
        resize_time(log_f0),
        resize_time(voiced),
    ]

    feat = np.vstack(parts).T.astype(np.float32)   # time x channels
    assert feat.shape == (TEMPORAL_FRAMES, len(TEMPORAL_FEATURE_NAMES))
    return feat

## 10. Global handcrafted descriptors

Global descriptors summarize duration, energy, spectral shape, pitch/voicing, spectral contrast and MFCC statistics.

In [ ]:
# ============================================================
# 10. Handcrafted global feature extraction
# ============================================================
def _stats(x, prefix, include_minmax=False):
    x = np.asarray(x, dtype=np.float32).ravel()
    x = x[np.isfinite(x)]
    if len(x) == 0:
        vals = {
            f"{prefix}_mean": 0.0,
            f"{prefix}_std": 0.0,
            f"{prefix}_median": 0.0,
            f"{prefix}_iqr": 0.0,
        }
        if include_minmax:
            vals.update({f"{prefix}_min":0.0, f"{prefix}_max":0.0})
        return vals
    vals = {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": float(np.std(x)),
        f"{prefix}_median": float(np.median(x)),
        f"{prefix}_iqr": float(np.percentile(x,75)-np.percentile(x,25)),
    }
    if include_minmax:
        vals.update({
            f"{prefix}_min": float(np.min(x)),
            f"{prefix}_max": float(np.max(x)),
        })
    return vals

def extract_global_handcrafted(y):
    d = {}
    d["duration_s"] = float(len(y) / SR)

    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH).flatten()
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH).flatten()
    centroid = librosa.feature.spectral_centroid(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH).flatten()
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH).flatten()
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, roll_percent=0.85).flatten()
    flatness = librosa.feature.spectral_flatness(y=y, n_fft=N_FFT, hop_length=HOP_LENGTH).flatten()

    d.update(_stats(rms, "rms", include_minmax=True))
    d.update(_stats(zcr, "zcr"))
    d.update(_stats(centroid, "centroid"))
    d.update(_stats(bandwidth, "bandwidth"))
    d.update(_stats(rolloff, "rolloff85"))
    d.update(_stats(flatness, "flatness"))

    # Pitch
    try:
        f0, voiced_flag, voiced_prob = librosa.pyin(
            y, fmin=PYIN_FMIN, fmax=PYIN_FMAX, sr=SR,
            frame_length=1024, hop_length=HOP_LENGTH
        )
        f0 = np.asarray(f0 if f0 is not None else [], dtype=np.float32)
        voiced_flag = np.asarray(voiced_flag if voiced_flag is not None else [], dtype=np.float32)
    except Exception:
        f0 = np.array([], dtype=np.float32)
        voiced_flag = np.array([], dtype=np.float32)

    voiced_f0 = f0[np.isfinite(f0) & (f0 > 0)]
    d.update(_stats(voiced_f0, "f0_hz", include_minmax=True))
    d["voiced_ratio"] = float(np.mean(voiced_flag > 0.5)) if len(voiced_flag) else 0.0

    # Log-Mel -> MFCC
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH, n_mels=N_MELS,
        fmin=FMIN, fmax=FMAX, power=2.0
    )
    logmel = librosa.power_to_db(mel + 1e-10, ref=np.max)
    mfcc = librosa.feature.mfcc(S=logmel, n_mfcc=N_MFCC)

    for i in range(N_MFCC):
        d[f"mfcc_{i:02d}_mean"] = float(np.mean(mfcc[i]))
        d[f"mfcc_{i:02d}_std"] = float(np.std(mfcc[i]))

    # Spectral contrast: 7 bands by default
    try:
        contrast = librosa.feature.spectral_contrast(
            y=y, sr=SR, n_fft=2048, hop_length=HOP_LENGTH
        )
        for i in range(contrast.shape[0]):
            d[f"contrast_{i:02d}_mean"] = float(np.mean(contrast[i]))
            d[f"contrast_{i:02d}_std"] = float(np.std(contrast[i]))
    except Exception:
        for i in range(7):
            d[f"contrast_{i:02d}_mean"] = 0.0
            d[f"contrast_{i:02d}_std"] = 0.0

    # Dynamic range proxies
    if len(rms):
        d["rms_p10"] = float(np.percentile(rms,10))
        d["rms_p90"] = float(np.percentile(rms,90))
        d["rms_dynamic_range"] = d["rms_p90"] - d["rms_p10"]
    else:
        d["rms_p10"] = d["rms_p90"] = d["rms_dynamic_range"] = 0.0

    return d

In [ ]:
# ============================================================
# 11. Extract temporal + handcrafted features
# ============================================================
temporal_list = []
global_rows = []

for row in tqdm(labels.itertuples(index=False), total=len(labels), desc="Acoustic features"):
    y, _, _, _ = load_audio(row.audio_path, SR)

    temp = extract_temporal(y)
    glob = extract_global_handcrafted(y)

    temporal_list.append(temp)
    glob["sample_id"] = row.sample_id
    global_rows.append(glob)

X_temporal_raw = np.stack(temporal_list).astype(np.float32)
global_df = pd.DataFrame(global_rows)

# Keep deterministic feature order, excluding sample_id.
GLOBAL_HANDCRAFTED_NAMES = [c for c in global_df.columns if c != "sample_id"]
X_handcrafted_raw = global_df[GLOBAL_HANDCRAFTED_NAMES].to_numpy(np.float32)

print("Temporal:", X_temporal_raw.shape)
print("Handcrafted:", X_handcrafted_raw.shape)

pd.DataFrame({"feature": TEMPORAL_FEATURE_NAMES}).to_csv(
    OUTPUT_ROOT / "temporal_feature_names.csv", index=False
)
pd.DataFrame({"feature": GLOBAL_HANDCRAFTED_NAMES}).to_csv(
    OUTPUT_ROOT / "global_handcrafted_feature_names.csv", index=False
)

global_df.to_csv(OUTPUT_ROOT / "global_handcrafted_raw.csv", index=False)

In [ ]:
# ============================================================
# 12. Train-only temporal normalization
# ============================================================
split_arr = labels["split"].to_numpy()
train_mask = split_arr == "train"

# Binary voiced channel remains binary.
VOICED_IDX = TEMPORAL_FEATURE_NAMES.index("voiced")
continuous_idx = [i for i in range(len(TEMPORAL_FEATURE_NAMES)) if i != VOICED_IDX]

temp_scaler = StandardScaler()
temp_scaler.fit(
    X_temporal_raw[train_mask][:,:,continuous_idx].reshape(-1, len(continuous_idx))
)

X_temporal = X_temporal_raw.copy()
all_cont = X_temporal_raw[:,:,continuous_idx].reshape(-1, len(continuous_idx))
X_temporal[:,:,continuous_idx] = temp_scaler.transform(all_cont).reshape(
    len(labels), TEMPORAL_FRAMES, len(continuous_idx)
)
X_temporal[:,:,VOICED_IDX] = (X_temporal_raw[:,:,VOICED_IDX] > 0.5).astype(np.float32)

joblib.dump(temp_scaler, OUTPUT_ROOT / "temporal_scaler_train_only.joblib")

print("Scaled temporal tensor:", X_temporal.shape)
print("Voiced unique values:", np.unique(X_temporal[:,:,VOICED_IDX]))

## 13. HuBERT embeddings

HuBERT is processed in deterministic 8-second chunks to keep memory bounded. Frame-level hidden states are pooled using the mean and standard deviation, resulting in a 1536-dimensional embedding per recording before PCA.

In [ ]:
# ============================================================
# 13. HuBERT model
# ============================================================
if EXTRACT_HUBERT:
    processor = Wav2Vec2FeatureExtractor.from_pretrained(HUBERT_MODEL_NAME)
    hubert = HubertModel.from_pretrained(HUBERT_MODEL_NAME).to(DEVICE)
    hubert.eval()

    @torch.no_grad()
    def hubert_embedding(y):
        chunk_len = int(HUBERT_CHUNK_SECONDS * SR)
        states = []

        for start in range(0, len(y), chunk_len):
            chunk = y[start:start+chunk_len]
            if len(chunk) < int(0.25 * SR):
                continue

            inp = processor(
                chunk,
                sampling_rate=SR,
                return_tensors="pt",
                padding=False
            )
            input_values = inp.input_values.to(DEVICE)

            if DEVICE.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    out = hubert(input_values).last_hidden_state
            else:
                out = hubert(input_values).last_hidden_state

            states.append(out.squeeze(0).float().cpu().numpy())

        if not states:
            return np.zeros(1536, dtype=np.float32)

        h = np.concatenate(states, axis=0)
        emb = np.concatenate([h.mean(axis=0), h.std(axis=0)], axis=0)
        return emb.astype(np.float32)

    hubert_embeddings = []
    for row in tqdm(labels.itertuples(index=False), total=len(labels), desc="HuBERT"):
        y, _, _, _ = load_audio(row.audio_path, SR)
        hubert_embeddings.append(hubert_embedding(y))

    X_hubert_raw = np.stack(hubert_embeddings).astype(np.float32)
    print("HuBERT raw:", X_hubert_raw.shape)

    np.save(OUTPUT_ROOT / "hubert_mean_std_raw.npy", X_hubert_raw)
else:
    X_hubert_raw = np.zeros((len(labels),0), dtype=np.float32)

In [ ]:
# ============================================================
# 14. Train-only global preprocessing and HuBERT PCA
# ============================================================
# Handcrafted scaler
hand_scaler = StandardScaler()
X_handcrafted = hand_scaler.fit_transform(
    X_handcrafted_raw[train_mask]
)
# Transform all after fitting on train
X_handcrafted = hand_scaler.transform(X_handcrafted_raw).astype(np.float32)
joblib.dump(hand_scaler, OUTPUT_ROOT / "handcrafted_scaler_train_only.joblib")

if EXTRACT_HUBERT:
    hubert_scaler = StandardScaler()
    hubert_scaler.fit(X_hubert_raw[train_mask])
    X_hubert_std = hubert_scaler.transform(X_hubert_raw)

    max_pca = min(
        HUBERT_PCA_DIM,
        int(train_mask.sum()) - 1,
        X_hubert_std.shape[1]
    )
    pca = PCA(n_components=max_pca, random_state=RANDOM_STATE)
    pca.fit(X_hubert_std[train_mask])
    X_hubert_pca = pca.transform(X_hubert_std).astype(np.float32)

    joblib.dump(hubert_scaler, OUTPUT_ROOT / "hubert_scaler_train_only.joblib")
    joblib.dump(pca, OUTPUT_ROOT / "hubert_pca_train_only.joblib")

    print("HuBERT PCA:", X_hubert_pca.shape)
    print("Explained variance:", float(pca.explained_variance_ratio_.sum()))
else:
    X_hubert_pca = np.zeros((len(labels),0), dtype=np.float32)

# Combined global representation
X_global_unscaled = np.concatenate(
    [X_handcrafted, X_hubert_pca], axis=1
).astype(np.float32)

final_global_scaler = StandardScaler()
final_global_scaler.fit(X_global_unscaled[train_mask])
X_global = final_global_scaler.transform(X_global_unscaled).astype(np.float32)
joblib.dump(final_global_scaler, OUTPUT_ROOT / "final_global_scaler_train_only.joblib")

GLOBAL_FEATURE_NAMES = (
    [f"handcrafted::{x}" for x in GLOBAL_HANDCRAFTED_NAMES] +
    [f"hubert_pca_{i:03d}" for i in range(X_hubert_pca.shape[1])]
)
pd.DataFrame({"feature": GLOBAL_FEATURE_NAMES}).to_csv(
    OUTPUT_ROOT / "global_feature_names.csv", index=False
)

print("Final global representation:", X_global.shape)

In [ ]:
# ============================================================
# 15. Participant-grouped 5-fold CV manifest within training set
# ============================================================
train_df = labels[labels["split"] == "train"].copy().reset_index()

sgkf = StratifiedGroupKFold(
    n_splits=N_CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_rows = []
for fold, (tr_idx, va_idx) in enumerate(
    sgkf.split(
        train_df,
        y=train_df["label"],
        groups=train_df["participant"]
    ),
    start=1
):
    fold_train = train_df.iloc[tr_idx]
    fold_val = train_df.iloc[va_idx]

    # Strict participant isolation check.
    assert set(fold_train["participant"]).isdisjoint(set(fold_val["participant"]))

    for role, part in [("train", fold_train), ("val", fold_val)]:
        for r in part.itertuples(index=False):
            cv_rows.append({
                "fold": fold,
                "role": role,
                "sample_id": r.sample_id,
                "participant": int(r.participant),
                "label": int(r.label),
                "class_code": r.class_code,
            })

cv_manifest = pd.DataFrame(cv_rows)
cv_manifest.to_csv(
    OUTPUT_ROOT / "participant_grouped_5fold_cv.csv", index=False
)

for fold in range(1, N_CV_FOLDS+1):
    a = cv_manifest[(cv_manifest.fold==fold) & (cv_manifest.role=="train")]
    b = cv_manifest[(cv_manifest.fold==fold) & (cv_manifest.role=="val")]
    print(
        f"Fold {fold}: train {len(a)} clips/{a.participant.nunique()} people | "
        f"val {len(b)} clips/{b.participant.nunique()} people | "
        f"val classes {b.label.value_counts().sort_index().to_dict()}"
    )

In [ ]:
# ============================================================
# 16. Save split NPZ files
# ============================================================
y_all = labels["label"].to_numpy(np.int64)
persons_all = labels["participant"].to_numpy(np.int64)
trials_all = labels["trial"].to_numpy(np.int64)
codes_all = labels["class_code"].astype(str).to_numpy()
ids_all = labels["sample_id"].astype(str).to_numpy()
audio_names_all = labels["audio_filename"].astype(str).to_numpy()

def save_split(split_name):
    m = labels["split"].to_numpy() == split_name

    payload = {
        "X_temporal": X_temporal[m].astype(np.float32),
        "X_temporal_raw": X_temporal_raw[m].astype(np.float32),
        "X_handcrafted": X_handcrafted[m].astype(np.float32),
        "X_global": X_global[m].astype(np.float32),
        "y": y_all[m],
        "persons": persons_all[m],
        "trials": trials_all[m],
        "class_codes": codes_all[m],
        "ids": ids_all[m],
        "audio_filenames": audio_names_all[m],
    }

    if EXTRACT_HUBERT:
        payload["X_hubert_raw"] = X_hubert_raw[m].astype(np.float32)
        payload["X_hubert_pca"] = X_hubert_pca[m].astype(np.float32)

    np.savez_compressed(
        OUTPUT_ROOT / f"{split_name}.npz",
        **payload
    )

    print(
        split_name,
        "n=", int(m.sum()),
        "temporal=", payload["X_temporal"].shape,
        "global=", payload["X_global"].shape,
        "classes=", Counter(payload["y"])
    )

for s in ["train","val","test"]:
    save_split(s)

In [ ]:
# ============================================================
# 17. Integrity audit
# ============================================================
audit = {
    "source_root": str(SOURCE_ROOT),
    "wav_dir": str(WAV_DIR),
    "source_label_file": str(LABEL_CSV),
    "num_samples": int(len(labels)),
    "num_participants": int(labels["participant"].nunique()),
    "class_mapping": {k:int(v) for k,v in source_mapping.items()},
    "class_counts": {
        str(int(k)): int(v)
        for k,v in labels["label"].value_counts().sort_index().items()
    },
    "split_counts": {},
    "participant_overlap": {
        "train_val": sorted(train_people & val_people),
        "train_test": sorted(train_people & test_people),
        "val_test": sorted(val_people & test_people),
    },
    "missing_wavs": missing,
    "ambiguous_wavs": ambiguous,
    "temporal_shape": list(X_temporal.shape),
    "global_shape": list(X_global.shape),
    "hubert_raw_shape": list(X_hubert_raw.shape),
    "hubert_pca_shape": list(X_hubert_pca.shape),
    "split_sha256": split_sha,
    "expected_split_sha256_match": split_sha == EXPECTED_SPLIT_SHA256,
}

for s in ["train","val","test"]:
    d = labels[labels["split"] == s]
    audit["split_counts"][s] = {
        "participants": int(d["participant"].nunique()),
        "samples": int(len(d)),
        "class_counts": {
            str(int(k)): int(v)
            for k,v in d["label"].value_counts().sort_index().items()
        }
    }

audit["passed"] = (
    len(missing) == 0 and
    len(ambiguous) == 0 and
    len(train_people & val_people) == 0 and
    len(train_people & test_people) == 0 and
    len(val_people & test_people) == 0 and
    split_sha == EXPECTED_SPLIT_SHA256 and
    len(labels) == EXPECTED_SAMPLES and
    labels["participant"].nunique() == EXPECTED_PARTICIPANTS
)

with open(OUTPUT_ROOT / "integrity_audit.json", "w") as f:
    json.dump(audit, f, indent=2)

print(json.dumps(audit, indent=2))
assert audit["passed"]

In [ ]:
# ============================================================
# 18. Dataset configuration
# ============================================================
config = {
    "dataset_name": "Participant-Independent WAV Acoustic Feature Dataset",
    "task": "three-class audio classification using source H/L/P codes",
    "sample_rate_hz": SR,
    "num_samples": EXPECTED_SAMPLES,
    "num_participants": EXPECTED_PARTICIPANTS,
    "class_mapping": EXPECTED_CLASS_MAPPING,
    "split_strategy": "frozen participant-independent train/validation/test",
    "split_sha256": split_sha,
    "temporal": {
        "frames": TEMPORAL_FRAMES,
        "channels": len(TEMPORAL_FEATURE_NAMES),
        "features": TEMPORAL_FEATURE_NAMES,
        "normalization": "train-only StandardScaler except voiced binary channel",
    },
    "global": {
        "handcrafted_dim": len(GLOBAL_HANDCRAFTED_NAMES),
        "hubert_model": HUBERT_MODEL_NAME if EXTRACT_HUBERT else None,
        "hubert_raw_dim": int(X_hubert_raw.shape[1]),
        "hubert_pca_dim": int(X_hubert_pca.shape[1]),
        "final_global_dim": int(X_global.shape[1]),
        "normalization": "train-only preprocessing and final StandardScaler",
    },
    "cv": {
        "strategy": "StratifiedGroupKFold by participant within frozen training participants",
        "n_splits": N_CV_FOLDS,
        "random_state": RANDOM_STATE,
    },
    "note": (
        "H, L and P are preserved as source class codes. "
        "No unsupported semantic class names are inferred."
    ),
}

with open(OUTPUT_ROOT / "dataset_config.json", "w") as f:
    json.dump(config, f, indent=2)

In [ ]:
# ============================================================
# 19. Example spectrogram figure
# ============================================================
sample_row = labels.iloc[0]
y, _, _, _ = load_audio(sample_row.audio_path, SR)

mel = librosa.feature.melspectrogram(
    y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH, n_mels=N_MELS,
    fmin=FMIN, fmax=FMAX, power=2.0
)
logmel = librosa.power_to_db(mel + 1e-10, ref=np.max)

plt.figure(figsize=(10,4))
librosa.display.specshow(
    logmel, sr=SR, hop_length=HOP_LENGTH,
    x_axis="time", y_axis="mel", fmax=FMAX
)
plt.colorbar(format="%+2.0f dB")
plt.title(
    f"Example Log-Mel Spectrogram | {sample_row.sample_id} | "
    f"code={sample_row.class_code}, label={sample_row.label}"
)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "example_logmel_spectrogram.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 20. Final package summary
# ============================================================
files = sorted([p.name for p in OUTPUT_ROOT.iterdir()])
print("Generated files:")
for f in files:
    print(" -", f)

print("\nTrain/Val/Test:")
for s in ["train","val","test"]:
    d = labels[labels["split"] == s]
    print(
        s,
        "| participants:", d["participant"].nunique(),
        "| samples:", len(d),
        "| classes:", d["label"].value_counts().sort_index().to_dict()
    )

print("\nTemporal shape:", X_temporal.shape)
print("Global shape:", X_global.shape)
print("Split SHA256:", split_sha)
print("Integrity audit passed:", audit["passed"])

In [ ]:
# ============================================================
# 21. Optional ZIP export to Google Drive
# ============================================================
ZIP_PATH = OUTPUT_ROOT.parent / "Audio_Engagement_Feature_Dataset_PI.zip"

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUTPUT_ROOT.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(OUTPUT_ROOT))

print("ZIP saved to:", ZIP_PATH)

## Recommended next modeling stage

After this notebook finishes, use `train.npz`, `val.npz`, and `test.npz` without changing the frozen participant split.

A strong next experiment is:

- Global branch: ExtraTrees / RandomForest / XGBoost on `X_global`
- Temporal branch: 1D-CNN, TCN, Transformer or Conformer on `X_temporal`
- Proposed model: tree-guided global-temporal fusion
- Primary metrics: accuracy, balanced accuracy, macro-F1, QWK, ordinal MAE if the three labels have a verified ordinal interpretation, ROC-AUC and log-loss
- Five-fold grouped CV only on the frozen training participants
- Validation is used for hyperparameter selection
- Test is touched only for the final evaluation

Do not use ordinal loss unless the source documentation confirms that H, L and P form a meaningful ordered scale.